### Week 6, Day 2 — Exercise solution

# My own MCP Server: push notifications

This is the exercise from the end of `2_lab2.ipynb`: make an MCP server with a simple function to send a push notification, then hook it up to an agent.

The server lives in `push_server.py` right next to this notebook. It follows exactly the same technique as `backend/accounts_server.py`:

1. Create a server object with `FastMCP("push_server")`
2. Decorate an ordinary Python function with `@mcp.tool()` — the function's type hints and docstring become the tool's schema and description, which is what the agent's LLM reads to decide when and how to call it
3. At the bottom, `mcp.run(transport='stdio')` starts the server, talking over stdin/stdout

The tool itself is the Pushover idea from earlier weeks of the course: a POST to `https://api.pushover.net/1/messages.json` with the `PUSHOVER_USER` and `PUSHOVER_TOKEN` from the `.env` file, plus the message.

In [ ]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
# On Windows, a stdio MCP server started from a Jupyter kernel writes to a stderr stream with no
# real file descriptor and crashes with io.UnsupportedOperation: fileno. We send the server's
# stderr to the null device so it always has somewhere real to write, which lets every cell below
# use MCPServerStdio exactly as the OpenAI Agents SDK documents it. Mac and Linux are unaffected.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

### First, use the server directly

Just like in the lab, we don't need an agent to talk to an MCP server. `MCPServerStdio` spawns `uv run push_server.py` as a subprocess (the notebook's working directory is this folder, so the relative path works), the MCP handshake happens over stdin/stdout, and we ask the server what tools it offers.

We should see one tool called `push`, with a schema derived automatically from the function signature and docstring.

In [ ]:
params = {"command": "uv", "args": ["run", "push_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

### Now, bring on the Agent!

We hand the running MCP server to an agent with `mcp_servers=[mcp_server]`. The agent sees the `push` tool, reads its description, and decides on its own to call it to satisfy the request. The LLM writes the message text — we never call `push()` ourselves.

In [ ]:
instructions = "You send push notifications to the user's phone when asked to notify them about something."
request = "I just built my first MCP server! Send me a short congratulatory push notification about it."
model = "gpt-5.4-mini"

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="push_agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("push_agent"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))